In [21]:
import pymupdf
from pathlib import Path
import pandas as pd

In [76]:
docs_path = Path('../data/pdfs')
images_path = Path('../data/images')
dfs_path = Path('../data/dfs')

In [82]:
df_docs = pd.read_csv(dfs_path / 'docs.csv', index_col='id')
df_docs.head(13)

,pdf_name,source
id,,
0,10-klas-istorija-ukrajiny-gisem-2018.pdf,Історія України (рівень стандарту) : підруч. д...
1,Istoriia Ukrainy_pidruchnyk dlia 7 klasu ZZSO_...,Історія України : підруч. для 7 класу закл. за...
2,Vstup-do-istorii-5-klas-Hisem-2022.pdf,Вступ до історії України та громадянської осві...
3,10-klas-istorija-ukrajini-vlasov-2018-prof.pdf,Історія України (профільний рівень) : підруч. ...
4,Istoria-Ukrainy_7kl_2024_Litera_ltd.pdf,Історія України : підруч. для 7 класу закл. за...
5,Istorija-Ukrajiny-8-klas-Halimov-2025.pdf,Історія України : підруч. для 8 класу закл. за...
6,Istoria-Ukraiiny-9-klas-Vlasov-2022-pohlyb.pdf,Історія України (поглиблений рівень) : підруч....
7,9-klas-istorija-ukrajini-gisem-2017.pdf,Історія України : підруч. для 9 класу закл. за...
8,Doslid-istoriyi-5-klas-Panarin-2022.pdf,Досліджуємо історію і суспільство : підруч. ін...


In [90]:
rows_text = []
rows_images = []
for index, doc_row in df_docs.iterrows():
    file_path = docs_path / doc_row['pdf_name']
    doc = pymupdf.open(file_path)

    for page in doc:
        page_data = page.get_text('dict')
        for block in page_data['blocks']:
            if block['type'] == 0:
                text = ' '.join(
                    span['text']
                    for line in block['lines']
                    for span in line['spans']
                )

                rows_text.append({
                    'text': text,
                    'bbox': tuple(block['bbox']),
                    'page': page.number,
                    'doc_id': index
                })

            elif block['type'] == 1:
                image_path = images_path / f"doc{index}_page{page.number}_{block['number']}.{block['ext']}"
                with open(image_path, "wb") as f:
                    f.write(block['image'])

                rows_images.append({
                    'image': str(image_path),
                    'bbox': tuple(block['bbox']),
                    'page': page.number,
                    'doc_id': index
                })

df_text = pd.DataFrame(rows_text)
df_images = pd.DataFrame(rows_images)

In [57]:
df_images.head(100)

,image,bbox,page,doc_id
0,../data/images/image_0.png,"(49.88980484008789, 208.3641357421875, 106.582...",1,0
1,../data/images/image_1.png,"(289.0788269042969, 96.16714477539062, 345.771...",1,0
2,../data/images/image_2.png,"(289.0788269042969, 397.1341247558594, 345.771...",1,0
3,../data/images/image_3.png,"(49.88980484008789, 586.6680908203125, 106.582...",1,0
4,../data/images/image_4.png,"(49.88980484008789, 264.8751220703125, 106.582...",1,0
5,../data/images/image_5.png,"(289.0788269042969, 453.8271179199219, 345.771...",1,0
6,../data/images/image_6.png,"(49.88980484008789, 643.3609619140625, 106.582...",1,0
7,../data/images/image_7.png,"(289.0788269042969, 583.9901123046875, 345.771...",1,0
8,../data/images/image_8.png,"(49.88980484008789, 321.25213623046875, 106.58...",1,0
9,../data/images/image_9.png,"(289.0788269042969, 208.18212890625, 345.77172...",1,0
